# Projet Probabilistic Graphical Model - NOT-MIWAE Vs MIWAE

### Run this if you work on Colab

In [ ]:
# Run if destination doesn't already exists
!git clone https://github.com/VincentGefflaut/notMIWAE-project.git

Cloning into 'notMIWAE'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 34 (delta 9), reused 8 (delta 8), pack-reused 19 (from 1)
Receiving objects: 100% (34/34), 34.92 KiB | 3.88 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/notMIWAE


In [ ]:
# Always run before working in Colab
import sys
sys.path.insert(0,'/content/notMIWAE-project')
!cd notMIWAE-project; git pull

In [ ]:
# Add path_prefix before all paths
path_prefix = '/content/notMIWAE-project/'

In [ ]:
# Cette commande remplace "import tensorflow as tf" par la version de compatibilité v1
# dans tous les fichiers .py pertinents.
!sed -i 's/import tensorflow as tf/import tensorflow.compat.v1 as tf; tf.disable_eager_execution()/g' MIWAE.py notMIWAE.py trainer.py task01.py

In [ ]:
import numpy as np
import pandas as pd
import tensorflow.compat.v1 as tf
# tf.disable_eager_execution() # Déjà géré par notre patch sed, mais sécurité
import os
import sys

# Imports du repo local
from MIWAE import MIWAE
from notMIWAE import notMIWAE
import trainer
import utils

from sklearn.preprocessing import StandardScaler

# --- Fonctions de chargement et préparation ---

def load_uci_data(dataset_name):
    """Charge les datasets via les URLs UCI."""
    print(f"Chargement de {dataset_name}...")
    if dataset_name == 'White':
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
        data = pd.read_csv(url, sep=';').values
    elif dataset_name == 'Red':
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
        data = pd.read_csv(url, sep=';').values
    elif dataset_name == 'Banknote':
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"
        data = pd.read_csv(url, header=None).values
    elif dataset_name == 'Concrete':
        # Concrete nécessite souvent openpyxl
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
        try:
            data = pd.read_excel(url).values
        except:
            print("Erreur chargement Excel Concrete. Installation dépendance...")
            os.system('pip install xlrd')
            data = pd.read_excel(url).values
    else:
        raise ValueError("Dataset non supporté dans ce script demo.")
    return data

def introduce_mnar_missing(X):
    """
    Simule le mécanisme MNAR (Self-masking) décrit dans l'article.
    Si la valeur > moyenne de la colonne, elle devient manquante.
    Appliqué sur la moitié des colonnes.
    """
    N, D = X.shape
    Xnan = X.copy()
    n_missing_cols = int(D / 2)

    # Calcul des moyennes par colonne
    means = np.mean(Xnan[:, :n_missing_cols], axis=0)

    # Création du masque (True = Manquant)
    mask_condition = Xnan[:, :n_missing_cols] > means

    # Application des NaN
    Xnan[:, :n_missing_cols][mask_condition] = np.nan

    # Version avec 0 à la place des NaN (pour l'input réseau)
    Xz = Xnan.copy()
    Xz[np.isnan(Xnan)] = 0

    return Xnan, Xz

# --- Paramètres de l'expérience (Tableau 1) ---

# Pour tester rapidement que tout marche, on réduit les itérations.
# Pour le VRAI résultat, mettez max_iter = 50000 ou 100000
MAX_ITER = 2000
BATCH_SIZE = 16
N_SAMPLES = 20
L_IMP = 500 # Échantillons pour l'imputation finale

# Liste des datasets à tester (Commencez par 'White' pour tester)
DATASETS = ['White', 'Banknote']

# Liste des modèles
MODELS = ['MIWAE', 'notMIWAE_selfmasking']

results = []

for dataset in DATASETS:
    try:
        # 1. Préparation Données
        data = load_uci_data(dataset)
        # On enlève la dernière colonne (target/class) comme dans task01.py
        data = data[:, :-1]

        N, D = data.shape
        dl = D - 1 # Dimension latente

        # Standardisation (Centrer-Réduire)
        scaler = StandardScaler()
        data = scaler.fit_transform(data)

        # Shuffle
        np.random.seed(42)
        p = np.random.permutation(N)
        data = data[p, :]

        Xtrain = data.copy()

        # Création des manques
        Xnan, Xz = introduce_mnar_missing(Xtrain)
        S = np.array(~np.isnan(Xnan), dtype=np.float32)

        # Pour simplifier, Xval = Xtrain ici (juste pour monitoring)
        Xval = Xtrain

        # 2. Entrainement des modèles
        for model_name in MODELS:
            tf.reset_default_graph() # Nettoyage mémoire TF
            sess_name = f"/tmp/{dataset}_{model_name}"

            print(f"\n--- Dataset: {dataset} | Modèle: {model_name} ---")

            if model_name == 'MIWAE':
                model = MIWAE(Xnan, Xval, n_latent=dl, n_samples=N_SAMPLES,
                              n_hidden=128, name=sess_name)
                trainer.train(model, batch_size=BATCH_SIZE, max_iter=MAX_ITER, name=sess_name)
                # Calcul RMSE
                rmse = utils.imputationRMSE(model, Xtrain, Xz, Xnan, S, L_IMP)[0]

            elif 'notMIWAE' in model_name:
                # Choix du processus (linear=agnostic, selfmasking, etc.)
                proc = 'linear' if 'agnostic' in model_name else 'selfmasking'
                if 'known' in model_name: proc = 'selfmasking_known'

                model = notMIWAE(Xnan, Xval, n_latent=dl, n_samples=N_SAMPLES,
                                 n_hidden=128, missing_process=proc, name=sess_name)
                trainer.train(model, batch_size=BATCH_SIZE, max_iter=MAX_ITER, name=sess_name)
                # Calcul RMSE
                rmse = utils.not_imputationRMSE(model, Xtrain, Xz, Xnan, S, L_IMP)[0]

            print(f"Résultat -> RMSE: {rmse:.4f}")
            results.append({'Dataset': dataset, 'Model': model_name, 'RMSE': rmse})

    except Exception as e:
        print(f"Erreur sur {dataset}: {e}")

print("\n=== RÉSULTATS FINAUX ===")
df = pd.DataFrame(results)
print(df)

Writing reproduce_table1.py


In [ ]:
# # Correction automatique de l'erreur "module 'numpy' has no attribute 'float'"
# # On remplace "np.float" par "float" en utilisant une expression régulière (\b = mot entier)
# # pour ne pas casser "np.float32" qui lui est toujours valide.

# !sed -i 's/np\.float\b/float/g' MIWAE.py notMIWAE.py task01.py utils.py trainer.py

# print("Patch appliqué avec succès. Vous pouvez relancer le script.")

Patch appliqué avec succès. Vous pouvez relancer le script.


In [ ]:
!python reproduce_table1.py

2025-11-19 19:31:57.113861: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763580717.134395    1807 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763580717.140645    1807 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763580717.156356    1807 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763580717.156382    1807 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763580717.156386    1807 computation_placer.cc:177] computation placer alr

# Explication de **reproduce_table1.py**

Le script `reproduce_table1.py` que nous avons créé est un **chef d'orchestre** : il ne contient pas la logique mathématique profonde des modèles, il se contente d'importer et d'utiliser les scripts originaux du GitHub que vous voyez dans votre capture d'écran.

Voici précisément comment `reproduce_table1.py` utilise les fichiers du dossier `notMIWAE` :

### 1\. Les Modèles (`MIWAE.py` et `notMIWAE.py`)

Dans `reproduce_table1.py`, les lignes suivantes chargent l'architecture des réseaux de neurones :

In [ ]:
from MIWAE import MIWAE
from notMIWAE import notMIWAE

* **`MIWAE.py`** : Fournit la classe `MIWAE`. Le script l'utilise pour créer le modèle de base (Encodeur/Décodeur gaussien).
  * **`notMIWAE.py`** : Fournit la classe `notMIWAE`. C'est ici que se trouve toute la logique spécifique à l'article, notamment le **mécanisme de données manquantes** (`missing_process='selfmasking'`) qui permet d'apprendre le lien entre la valeur d'une donnée et le fait qu'elle soit manquante.

### 2\. L'Entraînement (`trainer.py`)

Le script de reproduction appelle :

In [ ]:
import trainer
...
trainer.train(model, batch_size=BATCH_SIZE, ...)

* Il utilise **`utils.py`** pour faire l'imputation. C'est ce fichier qui contient les formules mathématiques complexes (Importance Sampling) pour estimer les valeurs manquantes proprement, comme décrit dans l'annexe de l'article.

### En résumé

Votre script `reproduce_table1.py` n'a fait que **préparer les données** (charger les vins, créer les trous MNAR) et **appeler les fonctions** qui étaient déjà codées dans les fichiers du GitHub.

C'est pour cela que nous avons dû appliquer les correctifs (`sed ...`) sur `MIWAE.py`, `notMIWAE.py`, `trainer.py` et `utils.py` avant de lancer le tout : si ces fichiers originaux avaient planté, votre script de reproduction aurait planté aussi puisqu'il dépend entièrement d'eux.